# Multi-Observation Grism Fitting Demo

This notebook demonstrates how to use geko's multi-observation fitting framework to jointly fit multiple grism observations.

## Overview

The multi-observation framework allows you to:
- Fit multiple R-dispersion observations at different position angles
- (Future) Fit R and C dispersion observations jointly
- Share galaxy parameters across observations while computing separate likelihoods

## Key Components

1. **GrismObservation**: Bundle for grism data (obs_map, obs_error, theta_rot, dispersion)
2. **run_inference_multi()**: MCMC runner for multiple observations
3. **compute_model_parametric_multi()**: Post-processing for multiple observations

In [ ]:
# Import necessary packages
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import numpyro

from geko import grism, models, utils
from geko.grism import GrismObservation
from geko.fitting import Fit_Numpyro
from astropy.modeling.models import Gaussian2D
from photutils.datasets import make_noise_image
from jax.scipy.signal import convolve

# Configure JAX and Numpyro
if 'gpu' in str(jax.devices()):
    print('Using GPU')
    numpyro.set_platform('gpu')
else:
    print('Using CPU')
    
numpyro.set_host_device_count(2)
numpyro.enable_validation()
jax.config.update('jax_enable_x64', True)

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Step 1: Create Mock Data

For this demo, we'll create the same observation twice to test the framework. In practice, you would have different observations at different angles or dispersion directions.

In [ ]:
def create_simple_psf(size=5, sigma=0.8):
    """Create a simple Gaussian PSF"""
    y, x = np.mgrid[0:size, 0:size]
    center = size // 2
    psf_model = Gaussian2D(amplitude=1, x_mean=center, y_mean=center,
                           x_stddev=sigma, y_stddev=sigma)
    psf = psf_model(x, y)
    psf = psf / psf.sum()  # Normalize
    return jnp.array(psf)

def create_mock_image(image_shape=31, PA=45.0, ellip=0.5, r_eff=4.0, SN=20):
    """Create a simple mock galaxy image"""
    print(f"Creating mock image: PA={PA}°, ellip={ellip}, r_eff={r_eff}")

    # Create coordinate grid
    x = jnp.linspace(0 - image_shape//2, image_shape - image_shape//2 - 1, image_shape)
    y = jnp.linspace(0 - image_shape//2, image_shape - image_shape//2 - 1, image_shape)
    x, y = jnp.meshgrid(x, y)

    # High-res version for convolution
    from jax import image
    x_grid = image.resize(x, (image_shape*5, image_shape*5), method='linear')
    y_grid = image.resize(y, (image_shape*5, image_shape*5), method='linear')

    # Create Sersic profile
    Ie = utils.flux_to_Ie(200, 1, r_eff, ellip)
    mock_image_highres = utils.compute_adaptive_sersic_profile(
        x_grid, y_grid, Ie/(5)**2, r_eff, 1, 0, 0, ellip,
        (90 - PA)*np.pi/180
    )
    mock_image = utils.resample(mock_image_highres, 5, 5)

    # Add PSF convolution
    psf = create_simple_psf()
    convolved_image = convolve(mock_image, psf, mode='same')

    # Add noise
    max_image = jnp.max(convolved_image)
    noise = make_noise_image(
        (image_shape, image_shape),
        distribution='gaussian',
        mean=0,
        stddev=max_image/SN
    )
    noisy_image = convolved_image + noise
    image_error = (max_image/SN) * jnp.ones((image_shape, image_shape))

    print(f"  Max flux: {max_image:.2f}, S/N: {SN}")

    return noisy_image, image_error, mock_image_highres, psf, convolved_image

In [ ]:
# Create mock galaxy image
image_shape = 31
PA_image = 45.0
ellip = 0.5
r_eff = 4.0

noisy_image, image_error, mock_image_highres, psf, convolved_image = create_mock_image(
    image_shape=image_shape, PA=PA_image, ellip=ellip, r_eff=r_eff, SN=20
)

# Plot the mock image
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

X = np.linspace(0 - image_shape//2, image_shape - image_shape//2 - 1, image_shape) * 0.06
Y = X

im0 = axs[0].imshow(utils.resample(mock_image_highres, 5, 5), origin='lower', cmap='PuBu')
axs[0].set_title('Mock Sersic Profile')
axs[0].set_xlabel('ΔRA [pixels]')
axs[0].set_ylabel('ΔDEC [pixels]')
plt.colorbar(im0, ax=axs[0])

im1 = axs[1].imshow(convolved_image, origin='lower', cmap='PuBu')
axs[1].set_title('+ PSF Convolution')
axs[1].set_xlabel('ΔRA [pixels]')
plt.colorbar(im1, ax=axs[1])

im2 = axs[2].imshow(noisy_image, origin='lower', cmap='PuBu')
axs[2].set_title('+ Gaussian Noise')
axs[2].set_xlabel('ΔRA [pixels]')
plt.colorbar(im2, ax=axs[2])

plt.tight_layout()
plt.show()

## Step 2: Create Grism Spectrum

Disperse the mock image through the grism with a velocity field.

In [ ]:
def create_grism_spectrum(mock_image_highres, psf, image_shape=31,
                          PA_grism=45.0, i=60.0, Va=150.0, r_t=3.0,
                          sigma0=50.0, SN_grism=15):
    """Create mock grism spectrum from image and velocity field"""
    print(f"Creating grism spectrum: PA={PA_grism}°, i={i}°, Va={Va} km/s")

    # Set up wavelength space
    wave_factor = 9
    delta_wave = 0.001
    wavelength = 4.5
    delta_wave_cutoff = 0.02
    wave_first = 4.0
    wave_space = jnp.linspace(wave_first, 5.0, int(1/delta_wave)+1)

    wave_min = wavelength - delta_wave_cutoff
    wave_max = wavelength + delta_wave_cutoff
    index_min = round((wave_min - wave_first)/delta_wave)
    index_max = round((wave_max - wave_first)/delta_wave)

    half_step = (delta_wave / wave_factor) * (wave_factor//2)
    wave_space_oversampled = np.arange(
        wave_space[0] - half_step,
        wave_space[-1] + delta_wave + half_step,
        delta_wave / wave_factor
    )

    # Initialize grism object
    factor = 5
    grism_object = grism.Grism(
        image_shape*factor, 0.0629/factor,
        icenter=image_shape//2, jcenter=image_shape//2,
        wavelength=wavelength, wave_space=wave_space_oversampled,
        index_min=index_min*wave_factor,
        index_max=(index_max+1)*wave_factor,
        grism_filter='F444W', grism_module='A', grism_pupil='R', PSF=psf
    )

    # Create velocity field
    from jax import image as jax_image
    x = jnp.linspace(0 - image_shape//2, image_shape - image_shape//2 - 1, image_shape)
    y = jnp.linspace(0 - image_shape//2, image_shape - image_shape//2 - 1, image_shape)
    x, y = jnp.meshgrid(x, y)
    x_grid = jax_image.resize(x, (image_shape*factor, image_shape*factor), method='linear')
    y_grid = jax_image.resize(y, (image_shape*factor, image_shape*factor), method='linear')

    kin_model = models.KinModels()
    V = kin_model.v(x_grid, y_grid, PA_grism, i, Va, r_t)
    D = sigma0 * jnp.ones_like(V)

    # Disperse
    grism_spectrum = grism_object.disperse(mock_image_highres, V, D)
    grism_spectrum = utils.resample(grism_spectrum, factor, wave_factor)

    # Add noise
    max_grism = jnp.max(grism_spectrum)
    grism_noise = make_noise_image(
        grism_spectrum.shape,
        distribution='gaussian',
        mean=0,
        stddev=max_grism/SN_grism
    )
    grism_spectrum_noise = grism_spectrum + grism_noise
    grism_error = (max_grism/SN_grism) * jnp.ones(grism_spectrum.shape)

    print(f"  Max grism flux: {max_grism:.2f}, S/N: {SN_grism}")

    return grism_spectrum_noise, grism_error, grism_object, grism_spectrum

In [ ]:
# Kinematic parameters
PA_grism = 45.0
i = 60.0
Va = 150.0
r_t = 3.0
sigma0 = 50.0

grism_spectrum_noise, grism_error, grism_object, grism_spectrum = create_grism_spectrum(
    mock_image_highres, psf, image_shape=image_shape,
    PA_grism=PA_grism, i=i, Va=Va, r_t=r_t, sigma0=sigma0, SN_grism=15
)

# Plot the grism spectrum
fig, axs = plt.subplots(2, 1, figsize=(8, 8))

X_wave = np.linspace(4.5 - 0.02, 4.5 + 0.02, grism_spectrum.shape[1])
Y_spatial = np.linspace(0 - image_shape//2, image_shape - image_shape//2 - 1, image_shape) * 0.06

im0 = axs[0].imshow(grism_spectrum, origin='lower', cmap='PuBu', aspect='auto')
axs[0].set_title('Mock 2D Grism Spectrum (no noise)')
axs[0].set_ylabel('ΔDEC [arcsec]')
plt.colorbar(im0, ax=axs[0])

im1 = axs[1].imshow(grism_spectrum_noise, origin='lower', cmap='PuBu', aspect='auto')
axs[1].set_title('+ Gaussian Noise')
axs[1].set_xlabel('Wavelength direction [pixels]')
axs[1].set_ylabel('ΔDEC [arcsec]')
plt.colorbar(im1, ax=axs[1])

plt.tight_layout()
plt.show()

## Step 3: Create GrismObservation Objects

The `GrismObservation` class bundles all the data for a single observation:
- grism: Grism object with dispersion physics
- obs_map: Observed 2D spectrum
- obs_error: Error map
- theta_rot: Rotation angle from prior reference frame
- dispersion: 'R' or 'C'

For this demo, we use the same observation twice to test the framework.

In [ ]:
# Create two GrismObservation objects
# In practice, these would be different observations (different angles or dispersion)
obs1 = GrismObservation(
    grism=grism_object,
    obs_map=grism_spectrum_noise,
    obs_error=grism_error,
    theta_rot=0.0,  # No rotation
    dispersion='R',
    name='obs1_theta0'
)

obs2 = GrismObservation(
    grism=grism_object,
    obs_map=grism_spectrum_noise,
    obs_error=grism_error,
    theta_rot=0.0,  # Same orientation (could be different)
    dispersion='R',
    name='obs2_theta0'
)

observations = [obs1, obs2]

print("Created GrismObservation objects:")
print(f"  {obs1}")
print(f"  {obs2}")

## Step 4: Set Up Kinematic Model and Priors

The kinematic model is shared across all observations.

In [ ]:
# Initialize kinematic model
kin_model = models.KinModels(
    im_shape=(image_shape, image_shape),
    pix_scale=0.06,
    redshift=5.0,
    filter='F444W'
)

# Set priors
priors = {
    'PA': PA_image,
    'i': i,
    'Va': Va,
    'r_t': r_t,
    'sigma0': sigma0,
    'n': 1.0
}
kin_model.disk.set_parametric_priors_test(priors)

print("Kinematic model initialized with priors:")
print(f"  PA = {PA_image}°")
print(f"  i = {i}°")
print(f"  Va = {Va} km/s")
print(f"  r_t = {r_t} pixels")
print(f"  sigma0 = {sigma0} km/s")

## Step 5: Run Multi-Observation MCMC Inference

The `run_inference_multi()` method:
1. Auto-generates masks for each observation (if not provided)
2. Samples galaxy parameters once in the prior reference frame
3. For each observation, applies rotation to PA and centroids
4. Computes model prediction and likelihood for each observation
5. Sums likelihoods to create joint posterior

In [ ]:
# Initialize Fit_Numpyro
fit = Fit_Numpyro(
    obs_map=grism_spectrum_noise,  # Still needed for mask generation
    obs_error=grism_error,
    grism_object=grism_object,
    kin_model=kin_model,
    inference_data=None,
    parametric=True
)

print("Running multi-observation MCMC inference...")
print("  This may take a few minutes...")
print("  (Using reduced samples for demo: 100 warmup, 100 samples, 2 chains)")

In [ ]:
# Run the multi-observation inference
fit.run_inference_multi(
    observations=observations,
    masks=None,  # Auto-generate masks
    num_samples=100,  # Reduced for demo (use 500+ for real analysis)
    num_warmup=100,
    num_chains=2,
    step_size=1,
    adapt_step_size=True,
    target_accept_prob=0.8
)

print("\nMCMC sampling completed!")
print(f"Posterior shape: {fit.inference_data.posterior.dims}")

## Step 6: Post-Process Results

The `compute_model_parametric_multi()` method generates model predictions for each observation.

In [ ]:
# Post-process results
results = kin_model.compute_model_parametric_multi(
    fit.inference_data,
    observations
)

print(f"Results computed for {len(results)} observations")
print(f"Observation names: {list(results.keys())}")
print(f"\nResult keys for each observation:")
for key in results[observations[0].name].keys():
    print(f"  - {key}")

## Step 7: Analyze Results

Check posterior statistics and compare with truth values.

In [ ]:
# Print posterior statistics
posterior = fit.inference_data.posterior
params = ['PA', 'i', 'Va', 'r_t', 'sigma0']

truth_vals = {
    'PA': PA_grism,
    'i': i,
    'Va': Va,
    'r_t': r_t,
    'sigma0': sigma0
}

print("Posterior Statistics (shared across observations)")
print("="*70)
print(f"{'Parameter':<15} {'Truth':<12} {'Median':<12} {'16%':<12} {'84%':<12}")
print("-"*70)

for param in params:
    samples = np.concatenate(posterior[param][:])
    median = np.percentile(samples, 50)
    p16 = np.percentile(samples, 16)
    p84 = np.percentile(samples, 84)
    truth = truth_vals[param]
    print(f"{param:<15} {truth:<12.2f} {median:<12.2f} {p16:<12.2f} {p84:<12.2f}")

## Step 8: Visualize Results

Plot model predictions vs observations for each observation.

In [ ]:
# Plot results for each observation
for obs_name, obs_results in results.items():
    obs = next(o for o in observations if o.name == obs_name)
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Results for {obs_name}', fontsize=16)
    
    # Observation
    im0 = axs[0].imshow(obs.obs_map, origin='lower', cmap='PuBu', aspect='auto')
    axs[0].set_title('Observed Grism Spectrum')
    axs[0].set_xlabel('Wavelength [pixels]')
    axs[0].set_ylabel('Spatial [pixels]')
    plt.colorbar(im0, ax=axs[0])
    
    # Model
    im1 = axs[1].imshow(obs_results['model_map'], origin='lower', cmap='PuBu', aspect='auto')
    axs[1].set_title('Model Prediction (median)')
    axs[1].set_xlabel('Wavelength [pixels]')
    plt.colorbar(im1, ax=axs[1])
    
    # Residuals
    residuals = (obs.obs_map - obs_results['model_map']) / obs.obs_error
    im2 = axs[2].imshow(residuals, origin='lower', cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
    axs[2].set_title('Residuals (σ)')
    axs[2].set_xlabel('Wavelength [pixels]')
    plt.colorbar(im2, ax=axs[2])
    
    plt.tight_layout()
    plt.show()

## Summary

This notebook demonstrated:

1. ✓ Creating `GrismObservation` objects to bundle observation data
2. ✓ Running multi-observation MCMC with `run_inference_multi()`
3. ✓ Post-processing with `compute_model_parametric_multi()`
4. ✓ Shared parameters sampled once across observations
5. ✓ Separate model predictions and likelihoods for each observation

## Next Steps

- Test with observations at different `theta_rot` angles
- (Future) Test with R and C dispersion observations jointly
- Use longer chains for convergence validation (500+ samples)
- Apply to real JWST data